In [1]:
import random

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [2]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [3]:
train_dataset = datasets.ImageFolder(root='data/train', transform=transform)
evaluation_dataset = datasets.ImageFolder(root='data/test/', transform = transform)

# Verify the loaded classes
print("Classes:", train_dataset.classes)
print("Training samples:", len(train_dataset))
print("Testing samples:", len(evaluation_dataset))


Classes: ['curly', 'straight']
Training samples: 800
Testing samples: 201


In [4]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [5]:
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [6]:
model  = HairModel()

In [7]:
trainable_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters

20073473

In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)


In [10]:
import json

epochs = 10

history = {
    "train_loss": [],
    "train_accuracy": [],
    "evaluation_loss": [],
    "evaluation_accuracy": []
}

for epoch in range(epochs):

    model.train()

    total_loss = 0.0
    correct = 0
    total_examples = 0

    for batch_x, batch_y in train_loader:

        batch_y = batch_y.float()

        logits = model(batch_x).squeeze(1)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = batch_y.size(0)

        total_loss += loss.item() * batch_size

        predictions = (torch.sigmoid(logits) >= 0.5)
        correct += (predictions == batch_y.bool()).sum().item()

        total_examples += batch_size

    train_loss = total_loss / total_examples
    train_accuracy = correct / total_examples

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    # Evaluation
    model.eval()

    evaluation_loss = 0.0
    evaluation_correct = 0
    evaluation_examples = 0

    with torch.no_grad():

        for batch_x, batch_y in evaluation_loader:

            batch_y = batch_y.float()

            logits = model(batch_x).squeeze(1)
            loss = criterion(logits, batch_y)

            batch_size = batch_y.size(0)

            evaluation_loss += loss.item() * batch_size

            predictions = (torch.sigmoid(logits) >= 0.5)
            evaluation_correct += (predictions == batch_y.bool()).sum().item()

            evaluation_examples += batch_size

    evaluation_loss /= evaluation_examples
    evaluation_accuracy = evaluation_correct / evaluation_examples

    history["evaluation_loss"].append(evaluation_loss)
    history["evaluation_accuracy"].append(evaluation_accuracy)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f} | "
        f"Eval Loss: {evaluation_loss:.4f} | "
        f"Eval Accuracy: {evaluation_accuracy:.4f}"
    )



Epoch 1/10 | Train Loss: 0.6638 | Train Accuracy: 0.6025 | Eval Loss: 0.6224 | Eval Accuracy: 0.6318
Epoch 2/10 | Train Loss: 0.5850 | Train Accuracy: 0.6875 | Eval Loss: 0.6284 | Eval Accuracy: 0.6567
Epoch 3/10 | Train Loss: 0.5267 | Train Accuracy: 0.7212 | Eval Loss: 0.6054 | Eval Accuracy: 0.6567
Epoch 4/10 | Train Loss: 0.4775 | Train Accuracy: 0.7688 | Eval Loss: 0.6833 | Eval Accuracy: 0.6119
Epoch 5/10 | Train Loss: 0.4235 | Train Accuracy: 0.7975 | Eval Loss: 0.5978 | Eval Accuracy: 0.7164
Epoch 6/10 | Train Loss: 0.3785 | Train Accuracy: 0.8325 | Eval Loss: 0.7184 | Eval Accuracy: 0.6368
Epoch 7/10 | Train Loss: 0.4149 | Train Accuracy: 0.8125 | Eval Loss: 0.6346 | Eval Accuracy: 0.6915
Epoch 8/10 | Train Loss: 0.2402 | Train Accuracy: 0.8925 | Eval Loss: 0.7856 | Eval Accuracy: 0.6965
Epoch 9/10 | Train Loss: 0.2164 | Train Accuracy: 0.9075 | Eval Loss: 0.7124 | Eval Accuracy: 0.7264
Epoch 10/10 | Train Loss: 0.1625 | Train Accuracy: 0.9325 | Eval Loss: 0.8356 | Eval Accura

In [11]:
with open("history_baseline.json", "w") as f:
    json.dump(history, f, indent=4)

In [12]:
median_accuracy = torch.tensor(history["train_accuracy"]).median().item()

print(f"Median Train Accuracy: {median_accuracy:.2f}")

Median Train Accuracy: 0.80


In [13]:
import json
import numpy as np
with open("history_baseline.json", "r") as f:
    history = json.load(f)

print(round(np.std(history["train_loss"], ddof=0),3))

0.156


### augmented

In [24]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [25]:
train_dataset.transform = train_transform_augmented

In [26]:
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

In [27]:
def train_model(model, optimizer, criterion, train_loader, evaluation_loader, epochs):

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        total_loss = 0.0
        correct = 0
        total_examples = 0

        for batch_x, batch_y in train_loader:

            batch_y = batch_y.float()

            logits = model(batch_x).squeeze(1)
            loss = criterion(logits, batch_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = batch_y.size(0)

            total_loss += loss.item() * batch_size

            predictions = (torch.sigmoid(logits) >= 0.5)
            correct += (predictions == batch_y.bool()).sum().item()

            total_examples += batch_size

        train_loss = total_loss / total_examples
        train_accuracy = correct / total_examples

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)

        # Evaluation
        model.eval()

        evaluation_loss = 0.0
        evaluation_correct = 0
        evaluation_examples = 0

        with torch.no_grad():

            for batch_x, batch_y in evaluation_loader:

                batch_y = batch_y.float()

                logits = model(batch_x).squeeze(1)
                loss = criterion(logits, batch_y)

                batch_size = batch_y.size(0)

                evaluation_loss += loss.item() * batch_size

                predictions = (torch.sigmoid(logits) >= 0.5)
                evaluation_correct += (predictions == batch_y.bool()).sum().item()

                evaluation_examples += batch_size

        evaluation_loss /= evaluation_examples
        evaluation_accuracy = evaluation_correct / evaluation_examples

        history["evaluation_loss"].append(evaluation_loss)
        history["evaluation_accuracy"].append(evaluation_accuracy)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Eval Loss: {evaluation_loss:.4f} | "
            f"Eval Accuracy: {evaluation_accuracy:.4f}"
        )

    return history

In [28]:
history_augmented = train_model(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    evaluation_loader=evaluation_loader,
    epochs=10
)

Epoch 1/10 | Train Loss: 0.4854 | Train Accuracy: 0.7750 | Eval Loss: 0.5526 | Eval Accuracy: 0.7164
Epoch 2/10 | Train Loss: 0.5233 | Train Accuracy: 0.7400 | Eval Loss: 0.6488 | Eval Accuracy: 0.6418
Epoch 3/10 | Train Loss: 0.4785 | Train Accuracy: 0.7525 | Eval Loss: 0.6065 | Eval Accuracy: 0.6667
Epoch 4/10 | Train Loss: 0.4814 | Train Accuracy: 0.7600 | Eval Loss: 0.5316 | Eval Accuracy: 0.7214
Epoch 5/10 | Train Loss: 0.4748 | Train Accuracy: 0.7762 | Eval Loss: 0.5154 | Eval Accuracy: 0.7264
Epoch 6/10 | Train Loss: 0.4926 | Train Accuracy: 0.7562 | Eval Loss: 0.5003 | Eval Accuracy: 0.7363
Epoch 7/10 | Train Loss: 0.4699 | Train Accuracy: 0.7638 | Eval Loss: 0.5617 | Eval Accuracy: 0.6766
Epoch 8/10 | Train Loss: 0.4362 | Train Accuracy: 0.7937 | Eval Loss: 0.6100 | Eval Accuracy: 0.6915
Epoch 9/10 | Train Loss: 0.4593 | Train Accuracy: 0.7600 | Eval Loss: 0.5167 | Eval Accuracy: 0.7363
Epoch 10/10 | Train Loss: 0.4660 | Train Accuracy: 0.7725 | Eval Loss: 0.5205 | Eval Accura

In [29]:
with open("history_augmented.json", "w") as f:
    json.dump(history, f, indent=4)

In [30]:
with open("history_augmented.json", "r") as f:
    history_augmented = json.load(f)

mean_eval_loss = np.mean(history_augmented["evaluation_loss"])

mean_eval_loss

np.float64(0.5910540987158296)

In [31]:
mean_last_five_accuracy = np.mean(
    history_augmented["evaluation_accuracy"][-5:]
)

mean_last_five_accuracy

np.float64(0.7034825870646766)